<a href="https://colab.research.google.com/github/SyedaMaryamFathima0105/titanic-ml-model/blob/main/Titanic_First_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

df.head()


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [4]:
df['Survived'].value_counts()

,count
Survived,
0,549
1,342


In [5]:
df=df.drop(['Name', 'Ticket', 'Cabin', 'PassengerId'], axis=1)

df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [6]:
df['Age'] = df['Age'].fillna(df['Age'].median())

df['Age'].isnull().sum()

np.int64(0)

In [7]:
df['Embarked'] = df['Embarked'].fillna('S')  # Fill 2 missing with 'S' = Southampton

df = pd.get_dummies(df, columns=['Sex', 'Embarked'], drop_first=True)

df.head()

,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_male,Embarked_Q,Embarked_S
0,0,3,22.0,1,0,7.2500,True,False,True
1,1,1,38.0,1,0,71.2833,False,False,False
2,1,3,26.0,0,0,7.9250,False,False,True
3,1,1,35.0,1,0,53.1000,False,False,True
4,0,3,35.0,0,0,8.0500,True,False,True


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X = df.drop('Survived', axis=1)  # Everything except the answer
y = df['Survived']               # The answer we want to predict

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

preds = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, preds))

Accuracy: 0.8100558659217877


In [9]:
# Make a fake passenger: Pclass=1, Age=22, SibSp=0, Parch=0, Fare=100, Sex_male=0, Embarked_Q=0, Embarked_S=0
fake_passenger = [[1, 22, 0, 0, 100, 0, 0, 0]]

prediction = model.predict(fake_passenger)
probability = model.predict_proba(fake_passenger)

print("Prediction:", "Survived" if prediction[0] == 1 else "Did not survive")
print("Survival chance:", round(probability[0][1] * 100, 1), "%")

Prediction: Survived
Survival chance: 96.0 %


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


In [10]:
import pandas as pd

feature_names = X.columns
importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': model.coef_[0]
}).sort_values('Importance', ascending=False)

print(importance)

      Feature  Importance
4        Fare    0.002591
1         Age   -0.030537
3       Parch   -0.108108
6  Embarked_Q   -0.113105
2       SibSp   -0.295244
7  Embarked_S   -0.398065
0      Pclass   -0.936817
5    Sex_male   -2.591503


In [11]:
  # Same passenger, but male: Sex_male=1 instead of 0
male_passenger = [[1, 22, 0, 0, 100, 1, 0, 0]]

prediction = model.predict(male_passenger)
probability = model.predict_proba(male_passenger)

print("Prediction:", "Survived" if prediction[0] == 1 else "Did not survive")
print("Survival chance:", round(probability[0][1] * 100, 1), "%")

Prediction: Survived
Survival chance: 64.3 %


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
